# Tool Registry 完全指南

**前置知识**: Python 装饰器、字典操作、上一节 Function Calling 内容

**学习目标**: 掌握工具的注册、管理、查询和导出

---

## 核心问题：如何管理多个工具？

上一节我们学会了定义单个函数，但实际 Agent 需要管理几十甚至上百个工具：

```
Agent 需要的能力:
├── 搜索类: web_search, wiki_search, code_search
├── 计算类: calculator, unit_convert, date_calc
├── 文件类: read_file, write_file, list_dir
└── API类: weather, stock, translate
```

**ToolRegistry 解决方案**：统一管理所有工具，支持注册、查询、启用/禁用、按标签过滤。

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import sys
import json
sys.path.insert(0, '..')

from src.tool_registry import (
    Tool,                    # 工具类：封装函数+元信息
    ToolRegistry,            # 注册表：管理多个工具
    tool,                    # 全局装饰器
    get_default_registry,    # 获取全局注册表
    reset_default_registry,  # 重置全局注册表
)
from src.function_calling import FunctionParameter, ParameterType

print("导入成功！")

---

## 第一步：理解 Tool 类

`Tool` = 函数 + 元信息（名称、描述、参数、标签等）

| 属性 | 作用 |
|------|------|
| `name` | 工具名称（LLM调用时使用） |
| `description` | 功能描述（帮助LLM选择） |
| `func` | 实际执行的函数 |
| `parameters` | 参数定义列表 |
| `tags` | 标签（用于分类过滤） |
| `enabled` | 是否启用 |
| `requires_confirmation` | 是否需要用户确认 |

In [ ]:
# ============================================================
# 手动创建 Tool 对象
# ============================================================

# 定义实际执行的函数
def add_numbers(a: int, b: int) -> int:
    """两数相加"""
    return a + b

# 创建 Tool 对象
add_tool = Tool(
    name="add",                      # 工具名
    description="计算两个整数的和",    # 描述
    func=add_numbers,                # 绑定的函数
    parameters=[                     # 参数定义
        FunctionParameter(name="a", type=ParameterType.INTEGER, description="第一个数"),
        FunctionParameter(name="b", type=ParameterType.INTEGER, description="第二个数"),
    ],
    tags=["math", "basic"],          # 标签：数学类、基础工具
)

# Tool 可以直接调用
result = add_tool(3, 5)  # 等价于 add_numbers(3, 5)
print(f"3 + 5 = {result}")

# 查看工具签名
print(f"签名: {add_tool.get_signature()}")

---

## 第二步：使用 ToolRegistry

手动创建 Tool 太繁琐，`ToolRegistry` 提供装饰器自动注册。

**核心方法**：
- `@registry.register` - 注册工具
- `registry.get(name)` - 按名称获取
- `registry.list_tools()` - 列出所有工具

In [ ]:
# ============================================================
# 使用装饰器注册工具
# ============================================================

# 创建注册表
registry = ToolRegistry()

# 方式1：最简注册（自动提取函数名和docstring）
@registry.register
def search(query: str, max_results: int = 10) -> str:
    """搜索信息"""
    return f"搜索'{query}'的前{max_results}条结果"

# 方式2：自定义名称和标签
@registry.register(name="weather", tags=["api", "external"])
def get_weather(city: str) -> str:
    """获取天气信息"""
    return f"{city}: 晴天, 25°C"

# 方式3：只指定标签
@registry.register(tags=["math"])
def calculate(expression: str) -> float:
    """计算数学表达式"""
    return eval(expression)

print(f"已注册 {len(registry)} 个工具")
print(f"工具列表: {registry.list_names()}")

---

## 第三步：查询工具

| 方法 | 作用 | 示例 |
|------|------|------|
| `get(name)` | 按名称获取，不存在返回None | `registry.get("search")` |
| `registry[name]` | 按名称获取，不存在抛异常 | `registry["search"]` |
| `name in registry` | 检查是否存在 | `"search" in registry` |
| `get_by_tag(tag)` | 按标签过滤 | `registry.get_by_tag("math")` |

In [ ]:
# ============================================================
# 查询工具
# ============================================================

# 按名称获取
search_tool = registry.get("search")
print(f"获取工具: {search_tool.name}")

# 索引语法（不存在会抛 KeyError）
weather_tool = registry["weather"]
print(f"索引获取: {weather_tool.name}")

# 检查存在性
print(f"'search' 存在: {'search' in registry}")
print(f"'unknown' 存在: {'unknown' in registry}")

# 按标签过滤
math_tools = registry.get_by_tag("math")
print(f"数学工具: {[t.name for t in math_tools]}")

api_tools = registry.get_by_tag("api")
print(f"API工具: {[t.name for t in api_tools]}")

# 列出所有标签
print(f"所有标签: {registry.list_tags()}")

---

## 第四步：启用/禁用工具

某些场景需要临时禁用工具（如：限制权限、调试）。

In [ ]:
# ============================================================
# 启用/禁用工具
# ============================================================

# 禁用工具
registry.disable("calculate")
print(f"calculate 启用状态: {registry.get('calculate').enabled}")

# 列出启用的工具（默认只返回启用的）
enabled_tools = registry.list_tools(enabled_only=True)
print(f"启用的工具: {[t.name for t in enabled_tools]}")

# 重新启用
registry.enable("calculate")
print(f"calculate 启用状态: {registry.get('calculate').enabled}")

---

## 第五步：导出为 API 格式

将注册表中的工具导出为 OpenAI/Anthropic API 格式，直接传给 LLM。

In [ ]:
# ============================================================
# 导出为 OpenAI 格式
# ============================================================
openai_tools = registry.to_openai_tools()
print(f"导出 {len(openai_tools)} 个工具")
print("\n第一个工具:")
print(json.dumps(openai_tools[0], indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 导出为 Anthropic 格式
# ============================================================
anthropic_tools = registry.to_anthropic_tools()
print("Anthropic 格式:")
print(json.dumps(anthropic_tools[0], indent=2, ensure_ascii=False))

---

## 第六步：生成工具描述

生成人类可读的工具描述，用于构建 prompt。

In [ ]:
# ============================================================
# 生成工具描述（用于 prompt）
# ============================================================
descriptions = registry.get_tool_descriptions()
print(descriptions)

---

## 第七步：全局装饰器 @tool

不想每次创建 ToolRegistry？使用全局 `@tool` 装饰器。

In [ ]:
# ============================================================
# 全局装饰器
# ============================================================

# 重置全局注册表（清空之前的注册）
reset_default_registry()

# 使用全局 @tool 装饰器
@tool
def global_search(query: str) -> str:
    """全局搜索"""
    return f"搜索: {query}"

@tool(name="global_calc", tags=["math"])
def my_calculator(expr: str) -> float:
    """全局计算器"""
    return eval(expr)

# 获取全局注册表
global_registry = get_default_registry()
print(f"全局注册表: {global_registry.list_names()}")

---

## 第八步：敏感操作确认

某些工具（删除文件、发邮件）需要用户确认后才能执行。

In [ ]:
# ============================================================
# 需要确认的工具
# ============================================================
sensitive_registry = ToolRegistry()

@sensitive_registry.register(requires_confirmation=True, tags=["dangerous"])
def delete_file(path: str) -> str:
    """删除文件（危险操作）"""
    return f"已删除: {path}"

@sensitive_registry.register(requires_confirmation=True)
def send_email(to: str, subject: str) -> str:
    """发送邮件"""
    return f"已发送到 {to}"

# 检查哪些工具需要确认
for t in sensitive_registry.list_tools():
    status = "需要确认" if t.requires_confirmation else "直接执行"
    print(f"{t.name}: {status}")

---

## 实战：构建完整工具集

In [ ]:
# ============================================================
# 完整示例：Agent 工具集
# ============================================================
from datetime import datetime

agent_tools = ToolRegistry()

# --- 搜索类 ---
@agent_tools.register(tags=["search"])
def web_search(query: str, num: int = 5) -> str:
    """搜索网页"""
    return f"网页搜索'{query}'返回{num}条结果"

@agent_tools.register(tags=["search"])
def wiki_search(topic: str) -> str:
    """搜索维基百科"""
    return f"维基百科: {topic}"

# --- 计算类 ---
@agent_tools.register(tags=["math"])
def calculator(expression: str) -> float:
    """计算数学表达式"""
    return eval(expression)

@agent_tools.register(tags=["math"])
def unit_convert(value: float, from_unit: str, to_unit: str) -> float:
    """单位转换"""
    conversions = {
        ("km", "mile"): 0.621371,
        ("mile", "km"): 1.60934,
    }
    factor = conversions.get((from_unit, to_unit), 1.0)
    return value * factor

# --- 时间类 ---
@agent_tools.register(tags=["time"])
def get_current_time() -> str:
    """获取当前时间"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 查看工具集
print(f"工具数量: {len(agent_tools)}")
print(f"标签: {agent_tools.list_tags()}")
print(f"\n搜索类: {[t.name for t in agent_tools.get_by_tag('search')]}")
print(f"数学类: {[t.name for t in agent_tools.get_by_tag('math')]}")

In [ ]:
# ============================================================
# 测试工具调用
# ============================================================
print("测试调用:")
print(f"  web_search: {agent_tools['web_search']('Python教程')}")
print(f"  calculator: {agent_tools['calculator']('2 ** 10')}")
print(f"  unit_convert: {agent_tools['unit_convert'](100, 'km', 'mile'):.2f} miles")
print(f"  get_current_time: {agent_tools['get_current_time']()}")

---

## 练习

创建文件操作工具集：`read_file`, `write_file`, `list_dir`，其中 `write_file` 需要确认。

In [ ]:
# ============================================================
# 练习：文件操作工具集
# ============================================================
file_tools = ToolRegistry()

# TODO: 实现以下工具
# @file_tools.register(tags=["file"])
# def read_file(path: str) -> str: ...

# @file_tools.register(tags=["file"], requires_confirmation=True)
# def write_file(path: str, content: str) -> str: ...

# @file_tools.register(tags=["file"])
# def list_dir(path: str) -> str: ...

---

## 本节要点

| 概念 | 作用 | 关键方法 |
|------|------|----------|
| `Tool` | 封装函数+元信息 | `get_signature()` |
| `ToolRegistry` | 管理多个工具 | `register()`, `get()`, `list_tools()` |
| `@tool` | 全局装饰器 | - |
| 标签系统 | 分类过滤 | `get_by_tag()`, `list_tags()` |
| 启用/禁用 | 动态控制 | `enable()`, `disable()` |
| API导出 | 对接LLM | `to_openai_tools()`, `to_anthropic_tools()` |

**下一步**: 学习 Structured Output，解析 LLM 的结构化输出。